In [0]:
import requests
import re
import zipfile
import os
import io
import urllib.request # ? 

In [0]:
# One target day for testing 2015-03-03
TARGET_DATE = "20150303"
# limits
MAX_FILES_TO_DOWNLOAD = 3
try: 
    MASTER_URL = dbutils.secrets.get(scope="default2", key="artem-library-link")
    display(dbutils.secrets.listScopes())
    display(dbutils.secrets.list(scope="default2"))
except Exception as e:
    print(f"Could not get secret: {e}")
# http://data.gdeltproject.org/gdeltv2/masterfilelist.txt (Link to library)
# Size(MB) /             MD5-hash                /      URL Link to download  /2015-02-18/minut/type          
# 150383 297a16b493de7cf6ca809a7cc31d0b93 http://data.gdeltproject.org/gdeltv2/20150218230000.export.CSV.zip

In [0]:
TARGET_URL = []

response = requests.get(MASTER_URL)
print("response status:", response.status_code)

if response.status_code == 200:
    lines = response.text.splitlines()
    print("Amount of lines: " + str(len(lines)))
    for line in lines:
        if TARGET_DATE in line and "export.CSV.zip" in line and MAX_FILES_TO_DOWNLOAD > len(TARGET_URL):
            url = line.split(" ")[2]
            print("Found: " + url)
            TARGET_URL.append(url)
  



In [0]:

VOLUME_PATH = "/Volumes/dbr_dev/artemzharkov10_bronze/raw_data/gdelt_history"

try:  
    os.makedirs(VOLUME_PATH, exist_ok=True)
except Exception as e:
    print(f"Could not create directory {VOLUME_PATH}: {e}")

for url in TARGET_URL:
    zip_filename = url.split('/')[-1] # 20150218230000.export.CSV.zip
    csv_filename = zip_filename.replace('.zip', '') # 20150218230000.export.CSV
    target_file_path = os.path.join(VOLUME_PATH, csv_filename) #create link to Azure
    # Indenpondence
    if os.path.exists(target_file_path):
        print(f" Data: {csv_filename} is already exist in Volume")
        continue #skip
     #=============================================
    response = requests.get(url)    
    try:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            # write in Azure  wb - write binary 
            with open(target_file_path, "wb") as f:
                file_to_write = z.open(csv_filename).read()
                f.write(file_to_write) 
    except Exception as e:
        print(f"Could not download file {url}: {e}")

In [0]:

df_raw = spark.read.option("delimiter", "\t").csv(VOLUME_PATH)
print(f"Count of raws in all downloaded files {df_raw.count()}")
display(df_raw.limit(10))